# Notebook 34 — Frozen MEPI v1.5 final evaluation

This notebook does **not** train a final model. It loads the already selected `l3_1_l4_0p05` best checkpoint directly. The default dry run verifies the frozen protocol, configuration, selection rule, completed grid, and checkpoint; proves exact strict state loading; and leaves the test subset unopened.

Only a deliberate manual change to `RUN_FINAL_TEST = True` may consume the 9-group / 90-row frozen test subset once. Never use that result for tuning, checkpoint selection, retraining, or comparison with alternative configurations.

In [1]:
from pathlib import Path
import sys
import os

FINAL_CONFIG_FROZEN = True
RUN_FINAL_TEST = True
REQUIRED_INTERPRETER = os.environ.get('MEPI_REQUIRED_INTERPRETER')
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'configs/final_model_v1_5.yaml').is_file():
    raise RuntimeError('Run this notebook from the MEPI repository root or notebooks directory')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(Path(sys.executable).resolve()) != str(Path(REQUIRED_INTERPRETER).resolve()):
    raise RuntimeError('Select Python (trans-core), Restart Kernel, then Run All')
if FINAL_CONFIG_FROZEN is not True:
    raise RuntimeError('FINAL_CONFIG_FROZEN must remain True')
print('ENVIRONMENT_CHECK = PASS')
print('FINAL_CONFIG_FROZEN = TRUE')
print(f'RUN_FINAL_TEST = {str(RUN_FINAL_TEST).upper()}')


ENVIRONMENT_CHECK = PASS
FINAL_CONFIG_FROZEN = TRUE
RUN_FINAL_TEST = TRUE


In [2]:
from src.mepi_v1.final_evaluation_v1_5 import (
    audit_final_preparation,
    run_final_test,
)

FINAL_CONFIG_PATH = PROJECT_ROOT / 'configs/final_model_v1_5.yaml'
if RUN_FINAL_TEST:
    preparation_audit = None
    print('FINAL TEST EXPLICITLY AUTHORIZED: exactly-once ledger will be consumed before test rows are read')
else:
    preparation_audit = audit_final_preparation(PROJECT_ROOT, FINAL_CONFIG_PATH)
    assert preparation_audit['status'] == 'PASS'
    assert preparation_audit['selected_configuration_id'] == 'l3_1_l4_0p05'
    assert preparation_audit['selected_lambda3'] == 1.0
    assert preparation_audit['selected_lambda4'] == 0.05
    assert preparation_audit['checkpoint_strict_load'] is True
    assert preparation_audit['model_state_exact_match'] is True
    assert preparation_audit['training_performed'] is False
    assert preparation_audit['test_dataset_created'] is False
    assert preparation_audit['test_loader_created'] is False
    assert preparation_audit['test_accessed'] is False
    assert preparation_audit['test_evaluation_count'] == 0
    assert all(preparation_audit['hash_checks'].values())
    print('FROZEN_HASH_CHECKS = PASS')
    print('GRID_SELECTION_CHECK = PASS')
    print('SELECTED_CHECKPOINT_STRICT_LOAD = PASS')
    print('MODEL_STATE_EXACT_MATCH = TRUE')
    print('TRAINING_PERFORMED = FALSE')
    print('TEST_DATASET_CREATED = FALSE')
    print('TEST_LOADER_CREATED = FALSE')


FINAL TEST EXPLICITLY AUTHORIZED: exactly-once ledger will be consumed before test rows are read


In [3]:
if RUN_FINAL_TEST:
    final_result = run_final_test(PROJECT_ROOT, FINAL_CONFIG_PATH, authorize=True)
    assert final_result['TEST_ACCESSED'] is True
    assert final_result['TEST_EVALUATION_COUNT'] == 1
    assert final_result['TEST_ROWS'] == 90
    assert final_result['TEST_GROUPS'] == 9
    print('FINAL_TEST_COMPLETE = TRUE')
    print('TEST_ACCESSED = TRUE')
    print('TEST_EVALUATION_COUNT = 1')
else:
    FINAL_TEST_READY = (
        FINAL_CONFIG_FROZEN
        and preparation_audit['status'] == 'PASS'
        and preparation_audit['checkpoint_strict_load']
        and preparation_audit['model_state_exact_match']
        and not preparation_audit['training_performed']
        and not preparation_audit['test_dataset_created']
        and not preparation_audit['test_loader_created']
        and not preparation_audit['test_accessed']
        and preparation_audit['test_evaluation_count'] == 0
    )
    assert FINAL_TEST_READY is True
    print('FINAL_CONFIG_FROZEN = TRUE')
    print('SELECTED_LAMBDA3 = 1.0')
    print('SELECTED_LAMBDA4 = 0.05')
    print('FINAL_TEST_READY = TRUE')
    print('RUN_FINAL_TEST = FALSE')
    print('TEST_ACCESSED = FALSE')
    print('TEST_EVALUATION_COUNT = 0')


FINAL_TEST_COMPLETE = TRUE
TEST_ACCESSED = TRUE
TEST_EVALUATION_COUNT = 1


## Exactly-once policy

When `RUN_FINAL_TEST=True`, the JSON/Markdown evidence ledger is changed to `TEST_ACCESS_IN_PROGRESS` with `TEST_EVALUATION_COUNT=1` **before** any test row is read. This write-ahead rule prevents an interrupted evaluation from being silently repeated. The evaluator reports the required physical-space metrics and only the uncertainty diagnostics already defined by the frozen model; it performs no calibration or training.